# Agentic_Mancala

CSCI 3202 Artificial Intelligence, Fall 2026, CU Boulder. Individual project, Option 2.

Development foundation: the instructor-provided HW 6 notebook, preserved byte for byte in `reference/Homework_6_Lastname_Firstname.ipynb`. The three course PDFs are local in `docs/` and excluded from public Git history.

**Task 2:** the game engine and random legal-move player are implemented. Section 3 contains the unchanged instructor examples/tests plus additional validation, and Section 4 contains seeded complete-game checks. Saved code outputs record the actual execution results. Instructor expected outputs remain labeled separately.

The 100-game intermediate-report experiment and analysis remain pending in Sections 5–6. Minimax, Alpha-Beta, and final-project experiments remain unimplemented. The README and AGENTS setup-status text describe the previous task; this notebook records the current implementation status.


## 1. Imports and configuration

Retain the instructor seed for reproducibility. Additional experiment configuration will be added only when implementation is authorized.


In [1]:
import random
random.seed(109)

## 2. Mancala game implementation

The instructor's board representation, constructor, display method, and public signatures are preserved. The four methods implement legal pit checks, random legal choices, sowing/captures/continuations, and terminal sweeping.

`valid_move` is a non-mutating check that rejects non-integer inputs (including booleans), empty pits, out-of-range pits, and terminal states. `random_move_generator` returns `None` when there is no legal move, without changing game state. `play` returns the existing board list on every path and records only successful moves. `winning_eval` returns a boolean and safely sweeps a terminal board even when called repeatedly; compare the two mancala totals to determine the winner or tie.

After a game-ending move, the player flag switches, matching the small-board instructor example. There is no next legal move or continuation on a terminal board.


## Mancala rules for this homework assignment and the course project
**(There are many different rule sets for Mancala. Please read this before writing the code.)**

* Players sit on opposite sides of the long edge of the board.
* There are 6 small pits in the middle of the board and 2 large ones at each end. The small ones on your side and the large pit on your right are yours. The small ones on the other side and the large pit to your opponent's right are theirs.
* The large pits at the ends of the board are called mancalas.
* Set up the board with 4 stones per small pit (none in the mancalas).
* On every turn, select a pit on your side of the board that contains one or more stones, then distribute its stones, one stone per pit, in a counter-clockwise direction until you have no stones remaining.
* If you encounter your opponent's mancala, skip it.
* If you encounter your mancala, drop a stone into it.
* **Continuation rule: if the last stone you drop lands in your own mancala, you take another turn immediately.** This can happen several times in a row. (The exception: if that move ends the game, there is no extra turn.)
* If the last stone lands in an empty pit on your side of the board, capture this stone and any stones in your opponent's pit on the other side of the board, collect all of these stones, including the one that just landed, and place them into your mancala. (If the opposite pit is empty, you still move your one stone into your mancala.)
* If either player's pits are entirely empty, the game concludes.
* The player who still has stones on their side of the board when the game concludes places all of these stones into their mancala.
* The player with the most stones in their mancala is declared the winner. If both players have an equal number of stones in their mancalas, the game is a tie.

In [2]:
class Mancala:
    def __init__(self, pits_per_player=6, stones_per_pit = 4):
        """
        The constructor for the Mancala class defines several instance variables:

        pits_per_player: This variable stores the number of pits each player has.
        stones_per_pit: It represents the number of stones each pit contains at the start of any game.
        board: This data structure is responsible for managing the Mancala board.
        current_player: This variable takes the value 1 or 2, as it's a two-player game, indicating which player's turn it is.
        moves: This is a list used to store the moves made by each player. It's structured in the format (current_player, chosen_pit).
        p1_pits_index: A list containing two elements representing the start and end indices of player 1's pits in the board data structure.
        p2_pits_index: Similar to p1_pits_index, it contains the start and end indices for player 2's pits on the board.
        p1_mancala_index and p2_mancala_index: These variables hold the indices of the Mancala pits on the board for players 1 and 2, respectively.
        """
        self.pits_per_player = pits_per_player
        self.board = [stones_per_pit] * ((pits_per_player+1) * 2)  # Initialize each pit with stones_per_pit number of stones 
        self.players = 2
        self.current_player = 1
        self.moves = []
        self.p1_pits_index = [0, self.pits_per_player-1]
        self.p1_mancala_index = self.pits_per_player
        self.p2_pits_index = [self.pits_per_player+1, len(self.board)-1-1]
        self.p2_mancala_index = len(self.board)-1
        
        # Zeroing the Mancala for both players
        self.board[self.p1_mancala_index] = 0
        self.board[self.p2_mancala_index] = 0

    def display_board(self):
        """
        Displays the board in a user-friendly format
        """
        player_1_pits = self.board[self.p1_pits_index[0]: self.p1_pits_index[1]+1]
        player_1_mancala = self.board[self.p1_mancala_index]
        player_2_pits = self.board[self.p2_pits_index[0]: self.p2_pits_index[1]+1]
        player_2_mancala = self.board[self.p2_mancala_index]

        print('P1               P2')
        print('     ____{}____     '.format(player_2_mancala))
        for i in range(self.pits_per_player):
            if i == self.pits_per_player - 1:
                print('{} -> |_{}_|_{}_| <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            else:    
                print('{} -> | {} | {} | <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            
        print('         {}         '.format(player_1_mancala))
        turn = 'P1' if self.current_player == 1 else 'P2'
        print('Turn: ' + turn)
        print()  # blank line between boards
        
    def valid_move(self, pit):
        """Return whether pit is a legal 1-indexed choice, without changing state."""
        # Reject non-integers (including bools) before using pit as an index.
        if not isinstance(pit, int) or isinstance(pit, bool):
            return False
        if not 1 <= pit <= self.pits_per_player:
            return False

        p1_start, p1_end = self.p1_pits_index
        p2_start, p2_end = self.p2_pits_index
        if not any(self.board[p1_start:p1_end + 1]) or not any(self.board[p2_start:p2_end + 1]):
            return False  # No legal moves once either side is empty.

        start = p1_start if self.current_player == 1 else p2_start
        return self.board[start + pit - 1] > 0
        
    def random_move_generator(self):
        """Choose uniformly from legal pits; return None if the game is over.

        Selection does not change the board, player, or move history. The caller
        controls random seeding so a complete sequence can be reproduced.
        """
        legal_pits = [pit for pit in range(1, self.pits_per_player + 1)
                      if self.valid_move(pit)]
        return random.choice(legal_pits) if legal_pits else None
    
    def play(self, pit):
        """Play one 1-indexed pit and return self.board.

        Check game over before invalid input. Only legal moves enter the history.
        Apply sowing, captures, the end-game sweep, and then the continuation rule.
        """
        if self.winning_eval():
            print("GAME OVER")
            return self.board
        if not self.valid_move(pit):
            print("INVALID MOVE")
            return self.board

        player = self.current_player
        if player == 1:
            start, end = self.p1_pits_index
            own_mancala = self.p1_mancala_index
            opponent_mancala = self.p2_mancala_index
        else:
            start, end = self.p2_pits_index
            own_mancala = self.p2_mancala_index
            opponent_mancala = self.p1_mancala_index

        index = start + pit - 1
        stones = self.board[index]
        self.board[index] = 0
        while stones > 0:
            index = (index + 1) % len(self.board)
            if index == opponent_mancala:
                continue
            self.board[index] += 1
            stones -= 1

        # One stone now means this own pit was empty just before the last drop.
        if start <= index <= end and self.board[index] == 1:
            opposite = len(self.board) - 2 - index
            self.board[own_mancala] += self.board[index] + self.board[opposite]
            self.board[index] = 0
            self.board[opposite] = 0

        self.moves.append((player, pit))
        game_over = self.winning_eval()
        if game_over or index != own_mancala:
            # Also switch after a game-ending move, as in the HW 6 example.
            # The terminal board has no next legal move, regardless of this flag.
            self.current_player = 3 - player
        return self.board
    
    def winning_eval(self):
        """Return a bool and sweep remaining stones when either side is empty.

        Repeated calls on a finished game are safe. Determine the winner by
        comparing the two mancala totals after this method returns True.
        """
        p1_start, p1_end = self.p1_pits_index
        p2_start, p2_end = self.p2_pits_index
        p1_stones = sum(self.board[p1_start:p1_end + 1])
        p2_stones = sum(self.board[p2_start:p2_end + 1])
        if p1_stones > 0 and p2_stones > 0:
            return False

        self.board[self.p1_mancala_index] += p1_stones
        self.board[self.p2_mancala_index] += p2_stones
        self.board[p1_start:p1_end + 1] = [0] * self.pits_per_player
        self.board[p2_start:p2_end + 1] = [0] * self.pits_per_player
        return True


## 3. Game-rule validation

The instructor's small-board code, expected-output Markdown, and eight-test cell below are unchanged. Their execution outputs are followed by a separate detailed expected/actual report and additional edge-case assertions. The diagnostic report never edits the original test fixtures. Every tested move is checked for stone conservation.


## Part 1: Small Board (3 Pits of 2 Stones each) (50 points)

For the first part of the assignment, you will work on a small Mancala board with 3 pits per player, each starting with 2 stones. Implement the following:

1. **play**: Implement the `play` function so that players take turns and make moves. It must distribute stones according to the rules above, perform captures, and apply the **continuation rule**: switch to the other player after a move *unless* the last stone landed in the current player's own mancala. **(20 points)**

2. **valid_move**: Implement the `valid_move` function to check that a player's chosen move is valid: the pit number must be in range and the pit must not be empty. **(15 points)**

3. **winning_eval**: Implement the `winning_eval` function to determine when the game is over. The game ends when either player's pits are all empty; the remaining stones go into their owner's mancala. The winner is the player with the most stones in their mancala; equal mancalas are a tie. **(15 points)**

Test your code with the sequence of moves in the Part 1 code cell below. Because of the continuation rule, the same player sometimes moves twice in a row, and the sequence also includes one invalid move and one move after the game has ended. Pits are 1-indexed when displaying and choosing a move. Your output must match the expected output shown below the code cell. Note the order of the checks in `play`: if the game is already over, print "GAME OVER" even if the chosen pit is also empty.

In [3]:
# Mancala part 1
game = Mancala(pits_per_player=3, stones_per_pit=2)
game.display_board()

# Player 1 selects pit 2: the last stone lands in P1's mancala, so P1 moves again
game.play(2)
game.display_board()

# Player 1 selects pit 1
game.play(1)
game.display_board()

# Player 2 selects pit 2: the last stone lands in P2's mancala, so P2 moves again
game.play(2)
game.display_board()

# Player 2 selects pit 1
game.play(1)
game.display_board()

# Player 1 selects pit 1, which is empty: INVALID MOVE, still Player 1's turn
game.play(1)
game.display_board()

# Player 1 selects pit 3
game.play(3)
game.display_board()

# Player 2 selects pit 2: extra turn again
game.play(2)
game.display_board()

# Player 2 selects pit 1: a capture that empties Player 1's side and ends the game
game.play(1)
game.display_board()

# Player 1 tries to move after the game has ended: GAME OVER
game.play(1)

# Printing the list of valid moves
print("\nList of valid moves:")
for move in game.moves:
    player, pit = move
    print(f"Player {player} selected pit {pit}")


P1               P2
     ____0____     
1 -> | 2 | 2 | <- 3
2 -> | 2 | 2 | <- 2
3 -> |_2_|_2_| <- 1
         0         
Turn: P1

P1               P2
     ____0____     
1 -> | 2 | 2 | <- 3
2 -> | 0 | 2 | <- 2
3 -> |_3_|_2_| <- 1
         1         
Turn: P1

P1               P2
     ____0____     
1 -> | 0 | 2 | <- 3
2 -> | 1 | 2 | <- 2
3 -> |_4_|_2_| <- 1
         1         
Turn: P2

P1               P2
     ____1____     
1 -> | 0 | 3 | <- 3
2 -> | 1 | 0 | <- 2
3 -> |_4_|_2_| <- 1
         1         
Turn: P2

P1               P2
     ____1____     
1 -> | 0 | 4 | <- 3
2 -> | 1 | 1 | <- 2
3 -> |_4_|_0_| <- 1
         1         
Turn: P1

INVALID MOVE
P1               P2
     ____1____     
1 -> | 0 | 4 | <- 3
2 -> | 1 | 1 | <- 2
3 -> |_4_|_0_| <- 1
         1         
Turn: P1

P1               P2
     ____1____     
1 -> | 0 | 5 | <- 3
2 -> | 1 | 2 | <- 2
3 -> |_0_|_1_| <- 1
         2         
Turn: P2

P1               P2
     ____2____     
1 -> | 0 | 6 | <- 3
2 -> | 1 | 0 | <-

#### Expected output for part 1

Your output should match this exactly, including the blank line after each board.

```
P1               P2
     ____0____     
1 -> | 2 | 2 | <- 3
2 -> | 2 | 2 | <- 2
3 -> |_2_|_2_| <- 1
         0         
Turn: P1

P1               P2
     ____0____     
1 -> | 2 | 2 | <- 3
2 -> | 0 | 2 | <- 2
3 -> |_3_|_2_| <- 1
         1         
Turn: P1

P1               P2
     ____0____     
1 -> | 0 | 2 | <- 3
2 -> | 1 | 2 | <- 2
3 -> |_4_|_2_| <- 1
         1         
Turn: P2

P1               P2
     ____1____     
1 -> | 0 | 3 | <- 3
2 -> | 1 | 0 | <- 2
3 -> |_4_|_2_| <- 1
         1         
Turn: P2

P1               P2
     ____1____     
1 -> | 0 | 4 | <- 3
2 -> | 1 | 1 | <- 2
3 -> |_4_|_0_| <- 1
         1         
Turn: P1

INVALID MOVE
P1               P2
     ____1____     
1 -> | 0 | 4 | <- 3
2 -> | 1 | 1 | <- 2
3 -> |_4_|_0_| <- 1
         1         
Turn: P1

P1               P2
     ____1____     
1 -> | 0 | 5 | <- 3
2 -> | 1 | 2 | <- 2
3 -> |_0_|_1_| <- 1
         2         
Turn: P2

P1               P2
     ____2____     
1 -> | 0 | 6 | <- 3
2 -> | 1 | 0 | <- 2
3 -> |_0_|_1_| <- 1
         2         
Turn: P2

P1               P2
     ____10____     
1 -> | 0 | 0 | <- 3
2 -> | 0 | 0 | <- 2
3 -> |_0_|_0_| <- 1
         2         
Turn: P1

GAME OVER

List of valid moves:
Player 1 selected pit 2
Player 1 selected pit 1
Player 2 selected pit 2
Player 2 selected pit 1
Player 1 selected pit 3
Player 2 selected pit 2
Player 2 selected pit 1
```

Player 2 wins, 10 to 2.

## Part 3: Show That Your Game Is Correct (20 points)

Run the test cell at the bottom of this notebook without changing it. It sets up eight board positions on the 6-pit board, plays a move (or two), and compares the board and the player to move next with the correct answer. The tests cover:

* the continuation rule for each player, including a move that goes all the way around the board and two extra turns in a row
* a move that does **not** earn an extra turn
* captures, including a capture when the opposite pit is empty
* a move whose last stone lands in your mancala but empties your side, which ends the game instead of giving an extra turn

The tests call `winning_eval()` after each move, so it must return `True` when the game is over and `False` otherwise, as its docstring says. Each passing test is worth 2.5 points. If a test fails, fix your `play` or `winning_eval` function rather than the test. You will reuse this game in the course project, where the same rules are checked again.

In [4]:
# Part 3: rule tests. Run this cell without changing it.
def run_test(name, board, player, pits, expected_board, expected_player, expected_over):
    game = Mancala()
    game.board = list(board)
    game.current_player = player
    for pit in pits:
        game.play(pit)
    over = game.winning_eval()
    ok = game.board == expected_board and over == expected_over and (expected_over or game.current_player == expected_player)
    print(('PASS  ' if ok else 'FAIL  ') + name)
    if not ok:
        print(f'      start {list(board)}, Player {player} plays pit(s) {pits}')
        print(f'      expected {expected_board}' + ('' if expected_over else f', then Player {expected_player} to move') + (', game over' if expected_over else ''))
        print(f'      got      {game.board}, Player {game.current_player} to move, game over = {over}')
    return ok

# board layout: [P1 pits 1-6, P1 mancala, P2 pits 1-6, P2 mancala]
tests = [
    ("Extra turn: P1 opens with pit 3, last stone lands in P1's mancala",
     [4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0], 1, [3],
     [4, 4, 0, 5, 5, 5, 1, 4, 4, 4, 4, 4, 4, 0], 1, False),
    ("No extra turn: P1 opens with pit 4, last stone lands on P2's side",
     [4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0], 1, [4],
     [4, 4, 4, 0, 5, 5, 1, 5, 4, 4, 4, 4, 4, 0], 2, False),
    ('Extra turn for Player 2',
     [4, 4, 4, 0, 5, 5, 1, 5, 4, 4, 4, 4, 4, 0], 2, [3],
     [4, 4, 4, 0, 5, 5, 1, 5, 4, 0, 5, 5, 5, 1], 2, False),
    ("Full lap: skip P2's mancala and finish in P1's mancala",
     [1, 1, 1, 1, 1, 14, 0, 1, 1, 1, 1, 1, 1, 0], 1, [6],
     [2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 0], 1, False),
    ('Two extra turns in a row (pit 6, then pit 5)',
     [3, 0, 0, 0, 2, 1, 0, 4, 4, 4, 4, 4, 4, 0], 1, [6, 5],
     [3, 0, 0, 0, 0, 1, 2, 4, 4, 4, 4, 4, 4, 0], 1, False),
    ('Capture: last stone in an empty P1 pit takes the opposite pit',
     [0, 1, 0, 0, 0, 3, 0, 4, 4, 4, 4, 4, 4, 0], 1, [2],
     [0, 0, 0, 0, 0, 3, 5, 4, 4, 4, 0, 4, 4, 0], 2, False),
    ('Capture when the opposite pit is empty: only the landing stone',
     [0, 1, 0, 0, 0, 3, 0, 4, 4, 4, 0, 4, 4, 0], 1, [2],
     [0, 0, 0, 0, 0, 3, 1, 4, 4, 4, 0, 4, 4, 0], 2, False),
    ("Last stone in own mancala empties P1's side: game ends, no extra turn",
     [0, 0, 0, 0, 0, 1, 10, 2, 2, 2, 2, 2, 2, 5], 1, [6],
     [0, 0, 0, 0, 0, 0, 11, 0, 0, 0, 0, 0, 0, 17], 1, True),
]

passed = sum(run_test(*t) for t in tests)
print(f'\n{passed} of {len(tests)} tests passed ({2.5 * passed} of 20 points)')


PASS  Extra turn: P1 opens with pit 3, last stone lands in P1's mancala
PASS  No extra turn: P1 opens with pit 4, last stone lands on P2's side
PASS  Extra turn for Player 2
PASS  Full lap: skip P2's mancala and finish in P1's mancala
PASS  Two extra turns in a row (pit 6, then pit 5)
PASS  Capture: last stone in an empty P1 pit takes the opposite pit
PASS  Capture when the opposite pit is empty: only the landing stone
PASS  Last stone in own mancala empties P1's side: game ends, no extra turn

8 of 8 tests passed (20.0 of 20 points)


In [5]:
# Separate diagnostics: the instructor's eight-test cell above stays unchanged.
assert passed == len(tests) == 8
for test_id, fixture in enumerate(tests, start=1):
    name, board, player, pits, expected_board, expected_player, expected_over = fixture
    tested_game = Mancala()
    tested_game.board = list(board)
    tested_game.current_player = player
    original_total = sum(board)
    for pit in pits:
        tested_game.play(pit)
        assert sum(tested_game.board) == original_total
        assert all(type(stones) is int and stones >= 0 for stones in tested_game.board)
    actual_over = tested_game.winning_eval()
    expected_next = None if expected_over else expected_player
    actual_next = None if actual_over else tested_game.current_player
    actual = (tested_game.board, actual_next, actual_over)
    expected = (expected_board, expected_next, expected_over)
    ok = actual == expected and type(actual_over) is bool
    print(f'T{test_id}: {name}')
    print(f'  Expected: board={expected_board}, next={expected_next}, over={expected_over}')
    print(f'  Actual:   board={tested_game.board}, next={actual_next}, over={actual_over}')
    print(f'  {"PASS" if ok else "FAIL"}; stones={sum(tested_game.board)}/{original_total}')
    assert ok, f'T{test_id} failed'
print('Terminal next=None means no legal next turn; the stored player flag is not graded by the terminal fixture.')


T1: Extra turn: P1 opens with pit 3, last stone lands in P1's mancala
  Expected: board=[4, 4, 0, 5, 5, 5, 1, 4, 4, 4, 4, 4, 4, 0], next=1, over=False
  Actual:   board=[4, 4, 0, 5, 5, 5, 1, 4, 4, 4, 4, 4, 4, 0], next=1, over=False
  PASS; stones=48/48
T2: No extra turn: P1 opens with pit 4, last stone lands on P2's side
  Expected: board=[4, 4, 4, 0, 5, 5, 1, 5, 4, 4, 4, 4, 4, 0], next=2, over=False
  Actual:   board=[4, 4, 4, 0, 5, 5, 1, 5, 4, 4, 4, 4, 4, 0], next=2, over=False
  PASS; stones=48/48
T3: Extra turn for Player 2
  Expected: board=[4, 4, 4, 0, 5, 5, 1, 5, 4, 0, 5, 5, 5, 1], next=2, over=False
  Actual:   board=[4, 4, 4, 0, 5, 5, 1, 5, 4, 0, 5, 5, 5, 1], next=2, over=False
  PASS; stones=48/48
T4: Full lap: skip P2's mancala and finish in P1's mancala
  Expected: board=[2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 0], next=1, over=False
  Actual:   board=[2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 0], next=1, over=False
  PASS; stones=25/25
T5: Two extra turns in a row (pit 6, then

### Additional engine edge cases

These assertions supplement the instructor tests. They check both player orientations, invalid inputs without mutation, small-board captures, multi-lap sowing, terminal sweeping and precedence, all three outcomes, legal seeded random choices, and conservation. Failure raises an assertion rather than printing an unconditional success.


In [6]:
from contextlib import redirect_stdout
from io import StringIO

edge_results = []

def check_equal(name, actual, expected):
    ok = actual == expected
    print(f'{"PASS" if ok else "FAIL"}: {name}; expected={expected!r}; actual={actual!r}')
    assert ok, name
    edge_results.append(name)

def snapshot(tested_game):
    return (list(tested_game.board), tested_game.current_player, list(tested_game.moves))

# Invalid input must leave board, player, and move history unchanged.
invalid_checks = 0
for pit_count in (3, 6):
    for player in (1, 2):
        tested_game = Mancala(pit_count, 2)
        tested_game.current_player = player
        for pit in (0, -1, pit_count + 1, 2 * pit_count + 1, 1.0, '1', None, True, False, [], {}):
            before = snapshot(tested_game)
            assert tested_game.valid_move(pit) is False
            output = StringIO()
            with redirect_stdout(output):
                returned = tested_game.play(pit)
            assert returned is tested_game.board
            assert output.getvalue() == 'INVALID MOVE\n'
            assert snapshot(tested_game) == before
            invalid_checks += 1
check_equal('Invalid inputs across two sizes and both players', invalid_checks, 44)

for player in (1, 2):
    tested_game = Mancala()
    tested_game.current_player = player
    start = 0 if player == 1 else 7
    tested_game.board[start] = 0
    before = snapshot(tested_game)
    output = StringIO()
    with redirect_stdout(output):
        returned = tested_game.play(1)
    assert returned is tested_game.board
    check_equal(f'P{player}: empty pit rejected without mutation',
                (tested_game.valid_move(1), output.getvalue(), snapshot(tested_game)),
                (False, 'INVALID MOVE\n', before))

tested_game = Mancala()
before = snapshot(tested_game)
check_equal('Active game returns bool False without mutation',
            (tested_game.winning_eval() is False, snapshot(tested_game)), (True, before))

# An empty side ends the game, even if the flagged player still has stones.
terminal_checks = 0
for empty_player in (1, 2):
    for player in (1, 2):
        tested_game = Mancala()
        tested_game.board = ([0] * 6 + [10] + [1] * 6 + [8] if empty_player == 1
                             else [1] * 6 + [10] + [0] * 6 + [8])
        tested_game.current_player = player
        total = sum(tested_game.board)
        before = snapshot(tested_game)
        assert not any(tested_game.valid_move(pit) for pit in range(1, 7))
        assert tested_game.random_move_generator() is None
        assert snapshot(tested_game) == before
        output = StringIO()
        with redirect_stdout(output):
            returned = tested_game.play(None)  # GAME OVER takes precedence over invalid input.
        assert returned is tested_game.board and output.getvalue() == 'GAME OVER\n'
        expected = ([0] * 6 + [10] + [0] * 6 + [14] if empty_player == 1
                    else [0] * 6 + [16] + [0] * 6 + [8])
        assert tested_game.board == expected and sum(tested_game.board) == total
        after = snapshot(tested_game)
        assert tested_game.winning_eval() is True
        assert tested_game.winning_eval() is True
        assert snapshot(tested_game) == after
        assert tested_game.current_player == player and tested_game.moves == []
        terminal_checks += 1
check_equal('Terminal precedence, no random move, sweep, and repeated evaluation', terminal_checks, 4)

# Expected boards are independent fixtures, not computed by the game engine.
extra_positions = [
    ('P2 capture', 6, [4,4,4,4,4,4,0,0,1,0,0,0,3,0], 2, [2],
     [4,4,4,0,4,4,0,0,0,0,0,0,3,5], 1, False),
    ('P2 capture with empty opposite pit', 6, [4,4,4,0,4,4,0,0,1,0,0,0,3,0], 2, [2],
     [4,4,4,0,4,4,0,0,0,0,0,0,3,1], 1, False),
    ('P2 full lap skips P1 mancala', 6, [1,1,1,1,1,1,0,1,1,1,1,1,14,0], 2, [6],
     [2,2,2,2,2,2,0,2,2,2,2,2,1,2], 2, False),
    ('P2 two consecutive extra turns', 6, [4,4,4,4,4,4,0,3,0,0,0,2,1,0], 2, [6,5],
     [4,4,4,4,4,4,0,3,0,0,0,0,1,2], 2, False),
    ('P1 multi-lap sowing', 6, [1,1,1,1,1,27,0,1,1,1,1,1,1,0], 1, [6],
     [3,3,3,3,3,2,3,3,3,3,3,3,3,0], 1, False),
    ('Capture in the just-emptied starting pit after a lap', 6,
     [1,1,1,1,1,13,0,1,1,1,1,1,1,0], 1, [6],
     [2,2,2,2,2,0,4,0,2,2,2,2,2,0], 2, False),
    ('No capture when landing in a nonempty own pit', 6, [4,4,4,4,4,4,0,4,4,4,4,4,4,0], 1, [1],
     [0,5,5,5,5,4,0,4,4,4,4,4,4,0], 2, False),
    ('No capture when landing in an empty opponent pit', 6,
     [0,0,0,0,1,2,0,0,3,0,0,0,0,0], 1, [6],
     [0,0,0,0,1,0,1,1,3,0,0,0,0,0], 2, False),
    ('Small-board P1 capture ends game', 3, [0,1,0,0,2,3,4,0], 1, [2],
     [0,0,0,3,0,0,0,7], 2, True),
    ('Small-board P2 capture ends game', 3, [2,3,4,0,0,1,0,0], 2, [2],
     [0,0,0,7,0,0,0,3], 1, True),
]
for name, pit_count, board, player, pits, expected_board, expected_player, expected_over in extra_positions:
    tested_game = Mancala(pit_count, 0)
    tested_game.board = list(board)
    tested_game.current_player = player
    total = sum(board)
    for pit in pits:
        mover = tested_game.current_player
        assert tested_game.valid_move(pit)
        returned = tested_game.play(pit)
        assert returned is tested_game.board and tested_game.moves[-1] == (mover, pit)
        assert sum(tested_game.board) == total
        assert all(type(stones) is int and stones >= 0 for stones in tested_game.board)
        if name == 'P2 two consecutive extra turns':
            assert tested_game.current_player == 2 and tested_game.winning_eval() is False
    check_equal(name, (tested_game.board, tested_game.current_player, tested_game.winning_eval()),
                (expected_board, expected_player, expected_over))

# Finish actual moves, then determine the outcome from the swept mancala totals.
outcome_positions = [
    ('P1 win', [0,0,1,7,1,1,1,0], 1, (8,3), 'P1'),
    ('P2 win', [0,0,1,0,2,2,2,0], 1, (1,6), 'P2'),
    ('Tie after P1 finishes in own mancala', [0,0,1,2,1,1,1,0], 1, (3,3), 'Tie'),
    ('Tie after P2 finishes in own mancala', [2,2,2,5,0,0,1,10], 2, (11,11), 'Tie'),
]
for name, board, player, expected_scores, expected_winner in outcome_positions:
    tested_game = Mancala(3, 0)
    tested_game.board = list(board)
    tested_game.current_player = player
    tested_game.play(3)
    assert tested_game.winning_eval() is True
    scores = (tested_game.board[3], tested_game.board[7])
    winner = 'Tie' if scores[0] == scores[1] else ('P1' if scores[0] > scores[1] else 'P2')
    assert sum(tested_game.board) == sum(board) == sum(scores)
    assert tested_game.current_player == 3 - player  # No continuation when game ends.
    assert tested_game.random_move_generator() is None
    check_equal(name, (scores, winner), (expected_scores, expected_winner))

saved_random_state = random.getstate()
try:
    for player, legal_pits in [(1, [2,5]), (2, [1,4,6])]:
        tested_game = Mancala()
        tested_game.current_player = player
        start = 0 if player == 1 else 7
        tested_game.board[start:start + 6] = [1 if pit in legal_pits else 0 for pit in range(1, 7)]
        before = snapshot(tested_game)
        random.seed(109)
        choices = [tested_game.random_move_generator() for _ in range(200)]
        assert all(pit in legal_pits for pit in choices)
        random.seed(109)
        assert choices == [tested_game.random_move_generator() for _ in range(200)]
        assert snapshot(tested_game) == before
        check_equal(f'P{player}: 200 legal random choices, reproducible and non-mutating',
                    sorted(set(choices)), legal_pits)
    tested_game = Mancala()
    tested_game.board[:6] = [0,0,0,0,0,1]
    check_equal('Random player with only one legal pit',
                [tested_game.random_move_generator() for _ in range(5)], [6] * 5)
finally:
    random.setstate(saved_random_state)

# Replay the instructor sequence, including the invalid and post-game attempts.
tested_game = Mancala(3, 2)
with redirect_stdout(StringIO()):
    for pit in [2,1,2,1,1,3,2,1,1]:
        tested_game.play(pit)
        assert sum(tested_game.board) == 12
check_equal('Small-board final board and valid-only history',
            (tested_game.board, tested_game.moves),
            ([0,0,0,2,0,0,0,10], [(1,2),(1,1),(2,2),(2,1),(1,3),(2,2),(2,1)]))
print(f'All {len(edge_results)} additional validation groups passed.')


PASS: Invalid inputs across two sizes and both players; expected=44; actual=44
PASS: P1: empty pit rejected without mutation; expected=(False, 'INVALID MOVE\n', ([0, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0], 1, [])); actual=(False, 'INVALID MOVE\n', ([0, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0], 1, []))
PASS: P2: empty pit rejected without mutation; expected=(False, 'INVALID MOVE\n', ([4, 4, 4, 4, 4, 4, 0, 0, 4, 4, 4, 4, 4, 0], 2, [])); actual=(False, 'INVALID MOVE\n', ([4, 4, 4, 4, 4, 4, 0, 0, 4, 4, 4, 4, 4, 0], 2, []))
PASS: Active game returns bool False without mutation; expected=(True, ([4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0], 1, [])); actual=(True, ([4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 0], 1, []))
PASS: Terminal precedence, no random move, sweep, and repeated evaluation; expected=4; actual=4
PASS: P2 capture; expected=([4, 4, 4, 0, 4, 4, 0, 0, 0, 0, 0, 0, 3, 5], 1, False); actual=([4, 4, 4, 0, 4, 4, 0, 0, 0, 0, 0, 0, 3, 5], 1, False)
PASS: P2 capture with empty opposite pit; ex

## 4. Random player

`random_move_generator` is implemented in the class in Section 2 and uses `random.choice` on the current player's legal pits. Seeding is controlled by the caller. The original Part 2 instructions and exercise placeholder are retained below; the student-chosen ten-move exercise is still available to complete.

The added complete-game validation uses seed 109 for a full-board demonstration, followed by 25 full-board games (seeds 109–133) and 10 small-board games (seeds 109–118). Each move checks legal selection, history, nonnegative integer counts, and conservation. A 10,000-move guard raises an error instead of silently accepting a nonterminating game. Random state is restored after each validation game.

These are engine validation runs. They do not calculate the intermediate report's 100-game statistics or first-player advantage.


## Part 2: Play Against a Random Player (6 Pits of 4 Stones each) (30 points)

In the second part of the assignment, use the full board: 6 pits with 4 stones in each pit. In addition to `play`, `valid_move` and `winning_eval`, create a random move generator for a random player.

1. **Random Move Generator**: Define `random_move_generator` so that it selects a random pit from the current player's non-empty pits. Set the `seed` value so that the generated values are consistent and reproducible when grading. **(15 points)**

You may refer to these links: [How to generate random integers in Python](https://machinelearningmastery.com/how-to-generate-random-numbers-in-python/#:~:text=Random%20integer%20values%20can%20be,for%20the%20generated%20integer%20values.), [How to use seed in Python random](https://www.w3schools.com/python/ref_random_seed.asp)

2. **Play a game**: Play **10** moves in total against the random player. You choose Player 1's moves and the random player chooses Player 2's. Because of the continuation rule, one player may make several moves in a row; always let `current_player` decide who moves next. Display the board after every move. **(10 points for playing the game, 5 points for printing the results)**

In [7]:
# Mancala part 2
# Create a 6-pit, 4-stone game, then play 10 moves: you choose Player 1's moves,
# random_move_generator() chooses Player 2's. Display the board after every move.


### Complete-game validation

The first run displays every chosen move and the final board. The following seeded batch checks normal termination on both supported board sizes. Outcomes are printed for the demonstration only; the intermediate-report statistics remain a separate task.


In [8]:
def validate_random_game(seed, pits_per_player=6, stones_per_pit=4, show_moves=False):
    saved_state = random.getstate()
    random.seed(seed)
    try:
        tested_game = Mancala(pits_per_player, stones_per_pit)
        initial_total = 2 * pits_per_player * stones_per_pit
        assert sum(tested_game.board) == initial_total
        for move_number in range(1, 10001):
            if tested_game.winning_eval():
                break
            mover = tested_game.current_player
            start, end = tested_game.p1_pits_index if mover == 1 else tested_game.p2_pits_index
            legal_pits = [i + 1 for i, stones in enumerate(tested_game.board[start:end + 1]) if stones > 0]
            before_choice = snapshot(tested_game)
            pit = tested_game.random_move_generator()
            assert pit in legal_pits and tested_game.valid_move(pit)
            assert snapshot(tested_game) == before_choice
            output = StringIO()
            with redirect_stdout(output):
                returned = tested_game.play(pit)
            assert output.getvalue() == ''  # A legal active move emits no error message.
            assert returned is tested_game.board
            assert tested_game.moves[-1] == (mover, pit) and len(tested_game.moves) == move_number
            assert all(type(stones) is int and stones >= 0 for stones in tested_game.board)
            assert sum(tested_game.board) == initial_total
            over = tested_game.winning_eval()
            if show_moves:
                next_turn = 'game over' if over else f'P{tested_game.current_player} next'
                print(f'Move {move_number:2d}: P{mover} pit {pit}; {next_turn}; stones={sum(tested_game.board)}')
            if over:
                break
        else:
            raise AssertionError(f'Seed {seed} exceeded the 10,000-move validation guard')

        assert tested_game.winning_eval() is True
        scores = (tested_game.board[tested_game.p1_mancala_index],
                  tested_game.board[tested_game.p2_mancala_index])
        assert sum(scores) == initial_total == sum(tested_game.board)
        assert all(stones == 0 for i, stones in enumerate(tested_game.board)
                   if i not in (tested_game.p1_mancala_index, tested_game.p2_mancala_index))
        assert tested_game.random_move_generator() is None
        final_state = snapshot(tested_game)
        assert tested_game.winning_eval() is True and snapshot(tested_game) == final_state
        if show_moves:
            tested_game.display_board()
            winner = 'Tie' if scores[0] == scores[1] else ('P1' if scores[0] > scores[1] else 'P2')
            print(f'COMPLETE: seed={seed}, moves={len(tested_game.moves)}, P1={scores[0]}, P2={scores[1]}, result={winner}')
        return tested_game
    finally:
        random.setstate(saved_state)

print('Full-board demonstration (seed 109):')
demo_game = validate_random_game(109, show_moves=True)
full_board_checks = [validate_random_game(seed) for seed in range(109, 134)]
small_board_checks = [validate_random_game(seed, 3, 2) for seed in range(109, 119)]
assert full_board_checks[0].moves == demo_game.moves and full_board_checks[0].board == demo_game.board
print(f'PASS: {len(full_board_checks)} full-board games, seeds 109–133; each conserved 48 stones and terminated.')
print(f'PASS: {len(small_board_checks)} small-board games, seeds 109–118; each conserved 12 stones and terminated.')
print('PASS: seed 109 replay produced the same moves and final board.')
print('No 100-game report statistics or first-player advantage were computed.')


Full-board demonstration (seed 109):
Move  1: P1 pit 3; P1 next; stones=48
Move  2: P1 pit 2; P2 next; stones=48
Move  3: P2 pit 4; P1 next; stones=48
Move  4: P1 pit 5; P2 next; stones=48
Move  5: P2 pit 5; P1 next; stones=48
Move  6: P1 pit 1; P1 next; stones=48
Move  7: P1 pit 2; P2 next; stones=48
Move  8: P2 pit 2; P2 next; stones=48
Move  9: P2 pit 6; P1 next; stones=48
Move 10: P1 pit 4; P2 next; stones=48
Move 11: P2 pit 3; P1 next; stones=48
Move 12: P1 pit 6; P2 next; stones=48
Move 13: P2 pit 5; P1 next; stones=48
Move 14: P1 pit 1; P2 next; stones=48
Move 15: P2 pit 2; P1 next; stones=48
Move 16: P1 pit 5; P2 next; stones=48
Move 17: P2 pit 6; P1 next; stones=48
Move 18: P1 pit 3; P2 next; stones=48
Move 19: P2 pit 4; P1 next; stones=48
Move 20: P1 pit 6; P2 next; stones=48
Move 21: P2 pit 2; P1 next; stones=48
Move 22: P1 pit 4; P1 next; stones=48
Move 23: P1 pit 5; P1 next; stones=48
Move 24: P1 pit 1; P2 next; stones=48
Move 25: P2 pit 3; P2 next; stones=48
Move 26: P2 p

## 5. Random-vs-random simulations

**Next task, pending separate authorization:** run the required 100-game full-board experiment with a documented seed and report Player 1 and Player 2 wins, losses, ties, and percentages; average moves per game; and average continuations per game. The smaller validation batch in Section 4 is not the intermediate-report experiment.

Count each pit-selection/sowing move, including extra-turn moves, in game length. Exclude game-ending moves from continuation counts. Follow `current_player` after every move. Distinguish player handoffs from individual moves. Do not infer first-player advantage from the validation batch.


## 6. Intermediate report results

**Report pending. Due October 30, 2026, 11:59 PM.** The engine and random player are implemented, with validation outputs in Sections 3–4. Remaining work is the 100-game experiment, player-specific win/loss/tie statistics, average move and continuation counts, first-player advantage analysis, and the notebook report. The retained Part 2 exercise also awaits the student's chosen Player 1 moves.

Describe actual results, libraries/frameworks, and any limitations after that work is authorized and executed. The course reference values (about 48% P1 wins, 45% P2 wins, 6% ties, 40.3 moves, and 5.5 continuations per game) are not this project's results. No intermediate-report statistics have been computed yet.


## 7. Minimax implementation

**Pending; no search code implemented.** Future interface: `minimax_player(game, state, d)` (or an equivalent class method), returning the 1-indexed pit for the player to move.

Use AI mancala minus opponent mancala as utility. Select MAX/MIN from the player to move in each state. Candidate moves reach depth 1, cutoff is `depth > d`, and every move counts, including continuation moves. Visit pits in ascending order and keep the first best move on ties.


## 8. Alpha-Beta implementation

**Pending; no search code implemented.** Future interface: `alpha_beta_player(game, state, d)` (or an equivalent class method). Apply pruning with the same utility, depth, continuation, and tie-breaking conventions as minimax. The selected pit must match minimax for the same state and depth.


## 9. AI experiments and performance analysis

**Pending implementation and execution.** Run 100 games for each required configuration: random vs. minimax at `d=2` and `d=5`; random vs. Alpha-Beta at `d=5` and `d=10`. Record actual win rates, game lengths, and required timings. Compare runtime at `d=5`, project minimax runtime at `d=10`, and plot AI win percentage at depths 2, 5, and 10. Explain the results.

Optional 20-point challenge: 100 Alpha-Beta games at `d=10` in under 20 minutes total, with at least 95% AI wins and all continuation checks passing. Use `time.perf_counter()` for total timing, record processor/memory, explain optimizations, and prepare a shorter interview demonstration. No measurements or graphs have been produced.


## 10. Continuation-rule verification

**Partially complete:** the engine rule tests and extra-turn edge cases are executed in Section 3. The final-project continuation average and both search algorithms remain pending. Re-run the unchanged eight HW 6 tests against the final game, report the random-game continuation average and counting convention, and test both algorithms on the positions below. Player 1 moves in each position.

Board layout: `[P1 pits 1-6, P1 mancala, P2 pits 1-6, P2 mancala]`. Depth uses the course's `depth > d` cutoff convention.

| Position | Board | Depth d | Instructor expected pit |
| --- | --- | --- | --- |
| A | `[7, 7, 1, 0, 2, 0, 5, 0, 9, 9, 1, 1, 2, 4]` | 1 | 5 |
| B | `[4, 10, 1, 4, 0, 1, 4, 1, 4, 4, 4, 4, 0, 7]` | 2 | 6 |
| C | `[3, 1, 1, 0, 2, 1, 10, 8, 2, 7, 0, 0, 0, 13]` | 2 | 6 |

These are required expected choices, not observed results. Report both algorithms' actual choices after implementation and explain how the search handles consecutive turns.


## 11. Final conclusions

**Pending. Final project due November 13, 2026.** Summarize actual findings, learned concepts, limitations, libraries/frameworks, and remaining questions once required experiments are complete. Prepare to explain the code and demonstrate alternate configurations during interview grading.
